In [37]:
import mlflow
# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Tech-Challenge-Telco-Churn")


<Experiment: artifact_location=('file:///c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/2'), creation_time=1777127798390, experiment_id='2', last_update_time=1777127798390, lifecycle_stage='active', name='Tech-Challenge-Telco-Churn', tags={}, workspace='default'>

In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature
import mlflow
import json
import joblib
import os
import mlflow
import matplotlib.pyplot as plt

In [39]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [ ]:
# --- 1. CONFIGURAÇÕES E SEPARAÇÃO ---
test_size = 0.2
val_size = 0.1  # 10% para validação (Early Stopping)
random_state = 42

cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

# Split 1: Treino+Val e Teste (O Teste fica isolado até o final)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# Split 2: Separando Validação do Treino
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_size, random_state=random_state, stratify=y_train_val
)

# --- 2. PRÉ-PROCESSAMENTO (Fitted apenas no Treino) ---
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_scaled = preprocessador.fit_transform(X_train)
X_val_scaled = preprocessador.transform(X_val)
X_test_scaled = preprocessador.transform(X_test)


# --- 3. PREPARAÇÃO PARA PYTORCH ---
def to_tensor(data): return torch.tensor(data, dtype=torch.float32)

X_train_t = to_tensor(X_train_scaled)
y_train_t = to_tensor(y_train.values).view(-1, 1)
X_val_t = to_tensor(X_val_scaled)
y_val_t = to_tensor(y_val.values).view(-1, 1)
X_test_t = to_tensor(X_test_scaled)
y_test_t = to_tensor(y_test.values).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)


# --- 4. ARQUITETURA MLP COM DROPOUT ---
class ChurnMLP(nn.Module):
    def __init__(self, input_size):
        super(ChurnMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3), # Regularização
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1) 
        )

    def forward(self, x):
        return self.net(x)
    
with torch.no_grad():
    logits = model(X_test_t)
    y_probs = torch.sigmoid(logits).numpy()
        
    # AJUSTE DE THRESHOLD: Focando em reduzir custo de Falso Negativo
    threshold = 0.30
    y_pred = (y_probs > threshold).astype(int)

# --- 5. TREINAMENTO COM PESO PARA CHURN ---
input_dim = X_train_scaled.shape[1]
model = ChurnMLP(input_dim)
# Calculando peso para tratar desbalanceamento (aumenta o Recall)
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
patience = 12
best_loss = np.inf
counter = 0

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()

    # Validação (Early Stopping)
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t)
    
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_mlp_model.pt')
    else:
        counter += 1
        if counter >= patience: break

# --- 1. CONFIGURAÇÕES DE CUSTO (Mantenha o padrão R$ 65) ---
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325

signature = infer_signature(X_test, y_pred)
model.load_state_dict(torch.load('best_mlp_model.pt'))
model.eval()

# --- 3. CÁLCULO DE MÉTRICAS ---
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# Cálculo de Custo de Negócio (Matriz de Confusão)
cm = confusion_matrix(y_test, y_pred)
# tn, fp, fn, tp = cm.ravel()
total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

os.makedirs("outputs", exist_ok=True)

csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(
    df,
    source=csv_path,
    name="dataset_normalizado_enginereed"
)

# --- 4. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="MLP_Neural_Network_v6"):

    mlflow.log_input(dataset, context="training/test")

    # Log de Tags
    mlflow.set_tags({
        "model_type": "mlp_neural_network",
        "model_architecture": "3-layers-128-64-32",
        "framework": "pytorch",
        "phase": "baseline",
        "dataset_name": "dataset_normalizado_enginereed",
        "dataset_size": "7043",
        "feature_set": "v2-clear-engineered"
    })

    # Log de Parâmetros
    mlflow.log_params({
        "model.hidden_layers": "[128,64,32]",
        "model.dropout_rate": 0.3,
        "model.activation": "relu",
        "train.optimizer": "Adam",
        "train.epochs": epochs,
        "train.learning_rate": 0.001,
        "train.batch_size": 64,
        "data.test_size": test_size,
        "data.random_seed": 42,
        "threshold.value": threshold,
    })

    # Log de Métricas
    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    # --- ARTEFATOS ---
    os.makedirs("outputs/mlp", exist_ok=True)

    # 1. Diagnostics: Matriz de Confusão
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Fica', 'Churn']
    ).plot(ax=ax, cmap='Blues')

    ax.set_title(f'Matriz de Confusão MLP (Threshold {threshold})')

    plot_path = "outputs/mlp/confusion_matrix_mlp.png"
    fig.savefig(plot_path)
    plt.close(fig)

    mlflow.log_artifact(plot_path, artifact_path="diagnostics")

    # 2. Preprocessor
    joblib.dump(preprocessador, "outputs/mlp/preprocessador.pkl")
    mlflow.log_artifact(
        "outputs/mlp/preprocessador.pkl",
        artifact_path="preprocessor"
    )

    # 3. Feature Names
    feature_names = list(X_train.columns)

    with open("outputs/mlp/feature_names.json", "w") as f:
        json.dump(feature_names, f)

    mlflow.log_artifact(
        "outputs/mlp/feature_names.json",
        artifact_path="data"
    )

    with open("outputs/mlp/classification_report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred))
        mlflow.log_artifact("outputs/mlp/classification_report.txt",
                        artifact_path="diagnostics")

    # 4. Model Card
    if os.path.exists("docs/model-card.md"):
        mlflow.log_artifact(
            "docs/model-card.md",
            artifact_path="model_card"
        )

    # 5. Log do Modelo PyTorch
    signature = infer_signature(X_test.head(5), y_pred[:5])

    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="modelo-pytorch",
        signature=signature,
        registered_model_name="mlp-pytorch-classifier-churn"
    )

print("-" * 30)
print("✅ MLP Finalizada e Registrada!")
print(f"Métricas: Recall {rec:.4f} | AUC {roc_auc:.4f} | Custo R$ {total_cost:.2f}")
print("-" * 30)

c:\Python311\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifact

------------------------------
✅ MLP Finalizada e Registrada!
Métricas: Recall 0.9278 | AUC 0.8474 | Custo R$ 37635.00
------------------------------
